# TURN-Mapper — train the learned segmenter on **Kaggle** (recommended: 30 free GPU-hours/week, 12-hour sessions)

Before running: **Settings → Accelerator → GPU T4 x2** (or P100) and **Settings → Internet → On** (needs a phone-verified Kaggle account). Then **Run All**. The CubiCasa5K zip (~5.5 GB) is downloaded from Zenodo inside the notebook.

Outputs (checkpoint, manifest, training log, eval) are written to `/kaggle/working/out` → after the run: **Output** tab → download, or "Save Version" to keep them.

CubiCasa5K: CC BY-NC 4.0 (non-commercial). Never commit the data or weights to git.

In [ ]:
import os, subprocess

OUT = "/kaggle/working/out"
os.makedirs(OUT, exist_ok=True)
!nvidia-smi -L

In [ ]:
!git clone -q https://github.com/ZakiNabeel/Turn-Mapper.git /kaggle/working/Turn-Mapper
%cd /kaggle/working/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1 || echo "tesseract optional for training"
!pip -q install -e ".[ml]"
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"

## 1. Download + extract CubiCasa5K (~10 min)

In [ ]:
!mkdir -p /tmp/turn
%cd /tmp/turn
!wget -q -O cubicasa5k.zip "https://zenodo.org/record/2613548/files/cubicasa5k.zip?download=1"
!unzip -q cubicasa5k.zip -d /tmp/turn/ && rm cubicasa5k.zip
!ls /tmp/turn/cubicasa5k | head
%cd /kaggle/working/Turn-Mapper

## 2. Convert SVG labels → masks (check a preview first)

In [ ]:
!turn-mapper ml prepare-cubicasa /tmp/turn/cubicasa5k --out /tmp/turn/cc_check --limit 10
import cv2, glob, matplotlib.pyplot as plt
from turn_mapper.learned.annotate import colorize

for p in sorted(glob.glob("/tmp/turn/cc_check/masks/*.png"))[:2]:
    m = cv2.imread(p, -1)
    g = cv2.imread(p.replace("masks", "images"), 0)
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(g, cmap="gray")
    plt.subplot(1, 2, 2)
    plt.imshow(colorize(m, g))
    plt.show()

In [ ]:
!turn-mapper ml prepare-cubicasa /tmp/turn/cubicasa5k --out /tmp/turn/cc_masks

## 3. Pre-train (≈ 3–5 h on a T4). Best checkpoint by validation mIoU is saved as it improves.

In [ ]:
!turn-mapper ml train --data /tmp/turn/cc_masks --out $OUT/turn_seg_cubicasa.pt \
    --epochs 30 --batch 8 --crop 512 --lr 3e-4 --workers 4 2>&1 | tee $OUT/train_stdout.txt

## 4. Evaluate on the CubiCasa test split

In [ ]:
!turn-mapper ml eval --data /tmp/turn/cc_masks --weights $OUT/turn_seg_cubicasa.pt --split test | tee $OUT/eval_cubicasa_test.json

## 5. Smoke-check the model inside the full pipeline (synthetic plan)

In [ ]:
import os

os.environ["TURN_SEG_WEIGHTS"] = f"{OUT}/turn_seg_cubicasa.pt"
!python scripts/demo_prepare.py --out /tmp/turn/demo > /dev/null
!turn-mapper run /tmp/turn/demo/lshape_greyfill.png --segmenter learned --out $OUT/learned_synthetic || true
!python eval/evaluate_synthetic.py --segmenter learned --out $OUT/eval_synth_learned | tail -16